# Module 3.1 — Tiền xử lý dữ liệu (Cửa hàng × Nhóm hàng × Ngày)

**Đầu vào:** `data/electronics-retail-transaction-q1-2025.xlsx` — Bảng kê bán hàng Genbyte, Quý 1/2025, toàn công ty.
**Code dùng chung:** `module3_preprocess.py` — các module khác `from module3_preprocess import doc_va_lam_sach, tao_bang_ngay, doc_bang_ngay, tao_dac_trung` để dùng đúng cùng một cách làm sạch, tổng hợp và tạo đặc trưng.
**Đầu ra (thư mục `outputs/`):**

| File | Dùng cho |
|---|---|
| `daily_store_category.csv` | **Bảng mô hình** cho cả 5 mô hình: mỗi dòng = 1 cửa hàng × 1 nhóm hàng × 1 ngày, đủ lịch (ngày không bán = 0) |
| `transactions_clean.csv` | Giao dịch đã làm sạch + cờ (bán buôn, khác, 0 đồng, trả lại) — để module khác tự lọc |
| `quality_report.csv` | Mục III — chất lượng dữ liệu |
| `data_dictionary.csv` | Giải thích cột của bảng mô hình |

Quy ước: **giữ tất cả giao dịch** (bán buôn, online, trả lại…), chỉ gắn cờ; muốn lọc thì truyền `bo_ban_buon=True`, `bo_khu_vuc_khac=True`, `bo_tra_lai=True`, `bo_0_dong=True` vào `tao_bang_ngay`. Chia Train–Validation–Test là phần của **Module 4**.

**Thứ tự chạy:** `Module3_1_tien_xu_ly` → `Module3_2_eda` → `Module3_3_random_forest`. Notebook 2 và 3 đọc `outputs/daily_store_category.csv`; nếu chưa có thì tự dựng lại từ file gốc.

**Cấu hình** (`.env`, mẫu ở `.env.example`): `DATA_FILE`, `DRIVE_ID`, `OUT_DIR`, `KHU_VUC`. Nếu chưa có file dữ liệu, module tự tải từ Google Drive theo `DRIVE_ID`.

**Chạy trên Google Colab:** upload `module3_preprocess.py` và `.env` lên cùng thư mục với notebook, rồi Runtime → Run all.

In [ ]:
import os
import pandas as pd
from module3_preprocess import DATA_FILE, OUT_DIR as OUT, KHU_VUC, KEY, doc_va_lam_sach, tao_bang_ngay, tai_ve

os.makedirs(OUT, exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Đọc và làm sạch dữ liệu (`doc_va_lam_sach`)
File export của Genbyte: 2 dòng tiêu đề trống, dòng 3 là tên cột, có dòng **Tổng cộng**. Hàm thực hiện:
1. Bỏ dòng tổng/dòng trống (không có ngày hoặc tên vật tư).
2. Chuẩn hoá kiểu: ngày, số; cắt khoảng trắng; ô trống → `"Không rõ"`.
3. **Nhóm hàng** = tên tiếng Việt của `Chủng loại` (mã lạ giữ nguyên mã và cảnh báo).
4. **Khu vực**: 13 khu vực shop bán lẻ; các mã không phải vùng shop (online, công ty/bán buôn, sửa chữa, kho, chi nhánh) gom thành nhóm **"Khác"**.
5. Gắn cờ thay vì xoá: bán buôn, khu vực Khác, dòng 0 đồng, tình trạng "Trả lại".
6. **Kiểm tra**: tổng Số lượng và Tổng đơn KH sau làm sạch phải bằng dòng *Tổng cộng* của file, lệch thì báo lỗi.

In [ ]:
df, info = doc_va_lam_sach(DATA_FILE, KHU_VUC)
print(f"Đọc {info['so_dong_doc']:,} dòng, bỏ {info['so_dong_bo']} dòng tổng/trống → còn {len(df):,} giao dịch | ô số bị thiếu: {info['so_thieu_so']}")
print(f"✔ Khớp dòng Tổng cộng: SL = {info['tong_sl_file']:,.0f} | Tổng đơn KH = {info['tong_dt_file']:,.0f}")
df.head()

## 2. Chất lượng dữ liệu (mục III)

In [ ]:
q = [
    ("Số dòng giao dịch", len(df)),
    ("Dòng tổng/trống đã bỏ", info["so_dong_bo"]),
    ("Khoảng ngày", f"{df['Ngày'].min():%d/%m/%Y} → {df['Ngày'].max():%d/%m/%Y}"),
    ("Số ngày lịch", (df["Ngày"].max() - df["Ngày"].min()).days + 1),
    ("Số ngày có giao dịch", df["Ngày"].nunique()),
    ("Số cửa hàng", df["Mã cửa hàng"].nunique()),
    ("Số khu vực (Mã KVKD)", df["Mã KVKD"].nunique()),
    ("Số nhóm hàng", df["Nhóm hàng"].nunique()),
    ("Số vật tư", df["Tên vật tư"].nunique()),
    ("Số nhân viên bán", df.loc[df["Tên NVBH"] != "Không rõ", "Tên NVBH"].nunique()),
    ("Dòng trùng hoàn toàn (giữ lại — có thể là mua 2 món giống nhau)", info["so_dong_trung"]),
    ("Ô số bị thiếu (đã điền 0)", info["so_thieu_so"]),
    ("Dòng số lượng âm", int((df["Số lượng"] < 0).sum())),
    ("Dòng tiền âm", int((df["Tổng đơn KH"] < 0).sum())),
    ("Dòng 0 đồng", int(df["la_0_dong"].sum())),
    ("Dòng bán buôn", int(df["la_ban_buon"].sum())),
    ("Dòng khu vực Khác (online/công ty/sửa chữa/kho)", int(df["la_khu_vuc_khac"].sum())),
    ("Dòng tình trạng 'Trả lại' (tính như bán, không trừ)", int(df["la_tra_lai"].sum())),
    ("Dòng thiếu Mã hạng (khách chưa là thành viên)", int(df["Mã hạng"].eq("Không rõ").sum())),
    ("Tổng số lượng", df["Số lượng"].sum()),
    ("Tổng đơn KH (VND)", df["Tổng đơn KH"].sum()),
]
quality = pd.DataFrame(q, columns=["Chỉ tiêu", "Giá trị"])
quality.to_csv(f"{OUT}/quality_report.csv", index=False, encoding="utf-8-sig")
display(quality)
for c in ["Loại hình bán", "Tình trạng", "Mã hạng", "Vùng"]:
    display(df.groupby(c).agg(Dòng=("Số lượng", "size"), SL=("Số lượng", "sum"), Doanh_thu=("Tổng đơn KH", "sum"))
              .sort_values("Dòng", ascending=False))

,Chỉ tiêu,Giá trị
0,Số dòng giao dịch,216995
1,Dòng tổng/trống đã bỏ,133
2,Khoảng ngày,01/01/2025 → 31/03/2025
3,Số ngày lịch,90
4,Số ngày có giao dịch,90
5,Số cửa hàng,130
6,Số khu vực (Mã KVKD),15
7,Số nhóm hàng,23
8,Số vật tư,3557
9,Số nhân viên bán,831


,Dòng,SL,Doanh_thu
Loại hình bán,,,
Bán lẻ,174267,"173,806.00",732419225073
Bán buôn,16862,"16,862.00",47327786800
Bán hàng Online qua sàn TMĐT,10090,"10,090.00",41442650583
Bán hàng Online tại Hoàng Hà,5922,"5,922.00",36684239001
Bán liên kết Mobifone,4825,"4,825.00",15011099500
Thu cũ đổi mới,2575,"2,575.00",29264357730
Bán dịch vụ,1478,"1,536.00",681754944
Trả nợ quà,634,634.00,0
Không rõ,342,342.00,119126000


,Dòng,SL,Doanh_thu
Tình trạng,,,
Hàng mới,171098,"171,098.00",843062287741
Không rõ,41293,"40,890.00",16641795344
Trôi bảo hành,1864,"1,864.00",9972766322
Hàng cũ,1249,"1,249.00",13945623600
Khách hàng đặt,907,907.00,16164497420
Trả lại,462,462.00,2567314204
Hàng lỗi,49,49.00,292237000
Trải nghiệm,41,41.00,139888000
Bảo hành,31,31.00,162045000


,Dòng,SL,Doanh_thu
Mã hạng,,,
Không rõ,139129,"138,883.00",551072283612
SILVER,40747,"40,650.00",162360002604
BRONZE,20407,"20,388.00",80476812610
GOLD,13183,"13,153.00",75766888319
DIAMOND,3529,"3,518.00",33274252486


,Dòng,SL,Doanh_thu
Vùng,,,
Miền Hà Nội,93547,"93,536.00",412067044564
Khác,31403,"31,405.00",119579128084
Miền Trung,28435,"28,306.00",110234794736
Miền Bắc,24585,"24,536.00",97454807971
Miền Nam,22629,"22,497.00",91038173006
Miền HCM,16396,"16,312.00",72576291270


## 3. Tổng hợp Cửa hàng × Nhóm hàng × Ngày (`tao_bang_ngay` — bảng mô hình)
- Cộng `Số lượng` (biến mục tiêu), `Tổng đơn KH`, số dòng nguồn, số dòng trả lại/bán buôn.
- **Lấp đủ lịch**: mỗi cặp cửa hàng × nhóm hàng từng phát sinh giao dịch có đủ mọi ngày trong kỳ; ngày không bán = 0 và `co_giao_dich = 0`.
- Hàm tự kiểm tra tổng số lượng/doanh thu không đổi và không có dòng trùng khoá.

In [ ]:
daily = tao_bang_ngay(df)
daily.to_csv(f"{OUT}/daily_store_category.csv", index=False, encoding="utf-8-sig")
n_cap, n_ngay = len(daily[KEY].drop_duplicates()), daily["Ngày"].nunique()
print(f"{n_cap:,} cặp cửa hàng × nhóm hàng × {n_ngay} ngày = {len(daily):,} dòng | tỷ lệ ô có giao dịch: {daily['co_giao_dich'].mean():.1%}")
daily.head()

## 4. Từ điển cột của bảng mô hình (gửi cả nhóm)


In [ ]:
dd = pd.DataFrame([
    ("Mã cửa hàng", "Mã shop ERP"), ("Nhóm hàng", "Tên tiếng Việt của Chủng loại"),
    ("Ngày", "Ngày chứng từ (Ngày CT)"),
    ("so_luong", "BIẾN MỤC TIÊU — tổng Số lượng bán trong ngày (0 nếu không bán)"),
    ("doanh_thu", "Tổng đơn KH (VND, gồm VAT)"), ("so_dong", "Số dòng giao dịch gốc"),
    ("so_dong_tra_lai", "Số dòng tình trạng 'Trả lại' (đã tính trong so_luong, không trừ)"),
    ("so_dong_ban_buon", "Số dòng bán buôn (đã tính trong so_luong)"),
    ("co_giao_dich", "1 = ô có giao dịch gốc, 0 = ô lấp lịch"),
    ("Mã KVKD", "Mã khu vực ERP"), ("Khu vực", "Khu vực; nhóm không phải shop ghi 'Khác - …'"),
    ("Vùng", "Miền Hà Nội / Miền Bắc / Miền Trung / Miền Nam / Miền HCM / Khác"),
], columns=["Cột", "Ý nghĩa"])
dd.to_csv(f"{OUT}/data_dictionary.csv", index=False, encoding="utf-8-sig")
df.drop(columns=["Ngày CT"]).to_csv(f"{OUT}/transactions_clean.csv", index=False, encoding="utf-8-sig")
display(dd); print("Đã lưu:", sorted(os.listdir(OUT)))

,Cột,Ý nghĩa
0,Mã cửa hàng,Mã shop ERP
1,Nhóm hàng,Tên tiếng Việt của Chủng loại
2,Ngày,Ngày chứng từ (Ngày CT)
3,so_luong,BIẾN MỤC TIÊU — tổng Số lượng bán trong ngày (...
4,doanh_thu,"Tổng đơn KH (VND, gồm VAT)"
5,so_dong,Số dòng giao dịch gốc
6,so_dong_tra_lai,Số dòng tình trạng 'Trả lại' (đã tính trong so...
7,so_dong_ban_buon,Số dòng bán buôn (đã tính trong so_luong)
8,co_giao_dich,"1 = ô có giao dịch gốc, 0 = ô lấp lịch"
9,Mã KVKD,Mã khu vực ERP


Đã lưu: ['by_category.csv', 'by_day.csv', 'by_store.csv', 'by_weekday.csv', 'coverage.csv', 'daily_store_category.csv', 'data_dictionary.csv', 'quality_report.csv', 'so_luong_theo_ngay.png', 'so_luong_theo_nhom_hang.png', 'transactions_clean.csv']


## 5. Tải kết quả về (Colab)

In [ ]:
tai_ve(OUT, "module3_tien_xu_ly_outputs")